# 14t VeRi-776 Fusion: CLIP-SENet v6 x TransReID 09v

Inference-only Kaggle experiment. Loads the canonical TransReID 09v VeRi checkpoint and the CLIP-SENet v6 kernel-output checkpoint, extracts VeRi-776 query/gallery features, sweeps score-level fusion and concat-feature fusion, and writes the full result grid plus verdict summary.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

print("Python:", sys.version)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "timm==1.0.11",
        "open_clip_torch==2.30.0",
        "pretrainedmodels==0.7.4",
        "loguru",
    ],
    check=True,
)

import numpy as np
import torch
import torch.nn.functional as F
import torchvision

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__)

if not Path("/kaggle/working/gp").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/MRKDaGods/gp.git", "/kaggle/working/gp"],
        check=True,
    )
if "/kaggle/working/gp" not in sys.path:
    sys.path.insert(0, "/kaggle/working/gp")

WORKING = Path("/kaggle/working")
FEATURE_DIR = WORKING / "14t_features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
"""CLIP-SENet architecture for vehicle re-identification.

Milestone M1 covers only the model port and local forward-pass smoke tests.
Training losses, camera/viewpoint embeddings, and Kaggle integration are
intentionally deferred.
"""

from __future__ import annotations

from dataclasses import dataclass

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger(__name__)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)
    logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class LoadedBackboneInfo:
    """Describes the backbone variant that was successfully loaded."""

    family: str
    model_name: str
    pretrained_tag: str | None = None


class AFEMBlock(nn.Module):
    """Adaptive Fine-grained Enhancement Module.

    This implements the paper's ambiguous Eq. (4) using the `(G + 1)`
    interpretation: `G` grouped weighted residual chunks plus one identity
    residual path. Set `residual_mode="sum_only"` to drop the identity term and
    return only the weighted grouped sum.
    """

    def __init__(
        self,
        in_dim: int = 2048,
        out_dim: int = 2048,
        num_groups: int = 32,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        if out_dim % num_groups != 0:
            raise ValueError(
                f"AFEM out_dim={out_dim} must be divisible by num_groups={num_groups}"
            )
        if residual_mode not in {"grouped_identity", "sum_only"}:
            raise ValueError(
                "residual_mode must be 'grouped_identity' or 'sum_only'"
            )

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_groups = num_groups
        self.group_dim = out_dim // num_groups
        self.residual_mode = residual_mode

        self.shared = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )
        self.group_weights = nn.Parameter(torch.randn(num_groups, self.group_dim))

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        linear = self.shared[0]
        nn.init.kaiming_normal_(linear.weight, mode="fan_out")
        bn = self.shared[1]
        nn.init.ones_(bn.weight)
        nn.init.zeros_(bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared(x)
        grouped = h.view(h.shape[0], self.num_groups, self.group_dim)
        weighted = grouped * self.group_weights.unsqueeze(0)
        enhanced = weighted.reshape(h.shape[0], self.out_dim)
        if self.residual_mode == "sum_only":
            return enhanced
        return h + enhanced


class _ResNetFeatureWrapper(nn.Module):
    """Wrap torchvision-style ResNet backbones to expose pooled 2048-d features."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return torch.flatten(x, 1)


class ResNet101IBNBranch(nn.Module):
    """Appearance branch backed by real ResNet101 IBN-a with deterministic fallbacks."""

    _IBN_MODEL = "resnet101_ibn_a"
    _FALLBACK_MODEL = "resnet101"

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.output_dim = 2048
        self.backbone: nn.Module
        self.loaded_backbone: LoadedBackboneInfo

        for loader in (
            self._load_pretrainedmodels_ibn,
            self._load_torch_hub_ibn,
            self._load_timm_ibn,
            self._load_timm_plain,
        ):
            loaded = loader(pretrained=pretrained)
            if loaded is None:
                continue
            self.backbone, self.loaded_backbone = loaded
            logger.info(
                "Appearance branch loaded via '%s' model='%s' pretrained_tag='%s'",
                self.loaded_backbone.family,
                self.loaded_backbone.model_name,
                self.loaded_backbone.pretrained_tag,
            )
            return

        raise ImportError(
            "Unable to load appearance backbone via pretrainedmodels, torch.hub, or timm"
        )

    def _load_pretrainedmodels_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import pretrainedmodels
        except ImportError:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' is unavailable; trying torch.hub"
            )
            return None

        constructor = getattr(pretrainedmodels, self._IBN_MODEL, None)
        if constructor is None:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' has no '%s' entry; trying torch.hub",
                self._IBN_MODEL,
            )
            return None

        pretrained_tag = "imagenet" if pretrained else None
        try:
            raw_model = constructor(pretrained=pretrained_tag)
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "last_linear"):
            raw_model.last_linear = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="pretrainedmodels",
            model_name=self._IBN_MODEL,
            pretrained_tag=pretrained_tag or "random_init",
        )

    def _load_torch_hub_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            raw_model = torch.hub.load(
                "XingangPan/IBN-Net",
                self._IBN_MODEL,
                pretrained=pretrained,
                trust_repo=True,
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'torch.hub' failed for '{}': {}",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "fc"):
            raw_model.fc = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="torch.hub",
            model_name=self._IBN_MODEL,
            pretrained_tag="official_pretrained" if pretrained else "random_init",
        )

    def _load_timm_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        available = set(timm.list_models())
        if self._IBN_MODEL not in available:
            logger.warning(
                "Appearance branch loader 'timm' has no '%s' entry; trying plain '%s'",
                self._IBN_MODEL,
                self._FALLBACK_MODEL,
            )
            return None

        try:
            backbone = timm.create_model(
                self._IBN_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._IBN_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def _load_timm_plain(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        try:
            backbone = timm.create_model(
                self._FALLBACK_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for plain '%s': %s",
                self._FALLBACK_MODEL,
                exc,
            )
            return None

        logger.warning(
            "Appearance branch fell back to plain '%s' because no IBN-a loader succeeded",
            self._FALLBACK_MODEL,
        )
        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._FALLBACK_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        if out.ndim != 2:
            raise RuntimeError(
                f"Appearance branch expected pooled 2D output, got shape {tuple(out.shape)}"
            )
        return out


class TinyCLIPImageBranch(nn.Module):
    """Semantic branch that loads TinyCLIP with a deterministic fallback chain."""

    _OPEN_CLIP_CHAIN = (
        {
            "model_name": "hf-hub:wkcn/TinyCLIP-ViT-45M-32-Text-21M-LAION400M",
            "pretrained_tag": None,
        },
        {
            "model_name": "TinyCLIP-ViT-40M-32-Text-19M",
            "pretrained_tag": "laion400m_e32",
        },
    )
    _TIMM_TINYCLIP_CHAIN = (
        "vit_medium_patch32_clip_224.tinyclip_laion400m",
    )
    _LAST_RESORT_OPEN_CLIP = ("ViT-B-32", "openai")

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.provider = ""
        self.model = None
        self.loaded_backbone: LoadedBackboneInfo | None = None
        last_error = self._try_load_open_clip(pretrained=pretrained)
        if self.model is None:
            last_error = self._try_load_timm_tinyclip(pretrained=pretrained) or last_error
        if self.model is None:
            last_error = self._try_load_open_clip_last_resort(pretrained=pretrained) or last_error

        if self.model is None or self.loaded_backbone is None:
            raise RuntimeError(
                "Unable to load any TinyCLIP/OpenCLIP visual backbone"
            ) from last_error

        self.image_size = self._infer_image_size(self.model)

    def _try_load_open_clip(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for candidate in self._OPEN_CLIP_CHAIN:
            model_name = candidate["model_name"]
            pretrained_tag = candidate["pretrained_tag"]
            try:
                if pretrained_tag is None:
                    model, _, _ = open_clip.create_model_and_transforms(model_name)
                else:
                    model, _, _ = open_clip.create_model_and_transforms(
                        model_name,
                        pretrained=pretrained_tag if pretrained else None,
                    )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP load failed for model='%s' pretrained='%s': %s",
                    model_name,
                    pretrained_tag or "hf-hub-default",
                    exc,
                )
                continue

            self.model = model
            self.provider = "open_clip"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag=pretrained_tag if pretrained else "random_init",
            )
            self.output_dim = self._infer_open_clip_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
                model_name,
                pretrained_tag if pretrained_tag is not None and pretrained else "hf-hub-default",
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_timm_tinyclip(self, pretrained: bool) -> Exception | None:
        try:
            import timm
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for model_name in self._TIMM_TINYCLIP_CHAIN:
            try:
                model = timm.create_model(
                    model_name,
                    pretrained=pretrained,
                    num_classes=0,
                )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP-equivalent timm load failed for model='%s': %s",
                    model_name,
                    exc,
                )
                continue

            self.model = model
            self.provider = "timm"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag="timm_pretrained" if pretrained else "random_init",
            )
            self.output_dim = self._infer_timm_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' via timm output_dim=%s",
                model_name,
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_open_clip_last_resort(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        model_name, pretrained_tag = self._LAST_RESORT_OPEN_CLIP
        try:
            model, _, _ = open_clip.create_model_and_transforms(
                model_name,
                pretrained=pretrained_tag if pretrained else None,
            )
        except Exception as exc:  # noqa: BLE001 - explicit last resort context
            logger.warning(
                "OpenCLIP last resort load failed for model='%s' pretrained='%s': %s",
                model_name,
                pretrained_tag,
                exc,
            )
            return exc

        self.model = model
        self.provider = "open_clip"
        self.loaded_backbone = LoadedBackboneInfo(
            family="semantic",
            model_name=model_name,
            pretrained_tag=pretrained_tag if pretrained else "random_init",
        )
        self.output_dim = self._infer_open_clip_output_dim(model)
        logger.info(
            "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
            model_name,
            pretrained_tag if pretrained else "random_init",
            self.output_dim,
        )
        return None

    @staticmethod
    def _infer_open_clip_output_dim(model: nn.Module) -> int:
        visual = getattr(model, "visual", None)
        output_dim = getattr(visual, "output_dim", None)
        if isinstance(output_dim, int):
            return output_dim

        visual_proj = getattr(model, "visual_projection", None)
        if isinstance(visual_proj, torch.Tensor) and visual_proj.ndim == 2:
            return int(visual_proj.shape[-1])

        visual_proj = getattr(visual, "proj", None)
        if isinstance(visual_proj, torch.Tensor):
            if visual_proj.ndim == 1:
                return int(visual_proj.shape[0])
            if visual_proj.ndim == 2:
                return int(visual_proj.shape[-1])

        raise RuntimeError("Could not infer TinyCLIP visual output dimension")

    @staticmethod
    def _infer_timm_output_dim(model: nn.Module) -> int:
        output_dim = getattr(model, "num_features", None)
        if isinstance(output_dim, int):
            return output_dim
        raise RuntimeError("Could not infer timm TinyCLIP visual output dimension")

    @staticmethod
    def _infer_image_size(model: nn.Module) -> tuple[int, int]:
        pretrained_cfg = getattr(model, "pretrained_cfg", None)
        if isinstance(pretrained_cfg, dict):
            input_size = pretrained_cfg.get("input_size")
            if isinstance(input_size, tuple) and len(input_size) == 3:
                return (int(input_size[-2]), int(input_size[-1]))

        visual = getattr(model, "visual", None)
        image_size = getattr(visual, "image_size", None)
        if isinstance(image_size, int):
            return (image_size, image_size)
        if isinstance(image_size, tuple) and len(image_size) == 2:
            return (int(image_size[0]), int(image_size[1]))
        return (224, 224)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if tuple(x.shape[-2:]) != self.image_size:
            x = F.interpolate(
                x,
                size=self.image_size,
                mode="bilinear",
                align_corners=False,
            )
        if self.provider == "open_clip":
            features = self.model.encode_image(x, normalize=False)
        else:
            features = self.model(x)
        if features.ndim != 2:
            raise RuntimeError(
                f"TinyCLIP branch expected 2D image features, got shape {tuple(features.shape)}"
            )
        return features


class CLIPSENet(nn.Module):
    """CLIP-SENet with a CNN appearance branch and a CLIP semantic branch."""

    def __init__(
        self,
        num_classes: int,
        embed_dim: int = 2048,
        afem_groups: int = 32,
        feat_dim_appearance: int = 2048,
        feat_dim_semantic: int = 512,
        dropout: float = 0.0,
        appearance_pretrained: bool = True,
        semantic_pretrained: bool = True,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        self.appearance_branch = ResNet101IBNBranch(pretrained=appearance_pretrained)
        self.semantic_branch = TinyCLIPImageBranch(pretrained=semantic_pretrained)

        detected_app_dim = self.appearance_branch.output_dim
        detected_sem_dim = self.semantic_branch.output_dim
        if feat_dim_appearance != detected_app_dim:
            logger.warning(
                "Requested feat_dim_appearance=%s but backbone reports %s. Using detected dim.",
                feat_dim_appearance,
                detected_app_dim,
            )
        if feat_dim_semantic != detected_sem_dim:
            logger.warning(
                "Requested feat_dim_semantic=%s but backbone reports %s. Using detected dim.",
                feat_dim_semantic,
                detected_sem_dim,
            )

        self.feat_dim_appearance = detected_app_dim
        self.feat_dim_semantic = detected_sem_dim
        self.fusion_fc = nn.Linear(
            self.feat_dim_appearance + self.feat_dim_semantic,
            embed_dim,
            bias=False,
        )
        self.afem = AFEMBlock(
            in_dim=embed_dim,
            out_dim=embed_dim,
            num_groups=afem_groups,
            residual_mode=residual_mode,
        )
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

        nn.init.kaiming_normal_(self.fusion_fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.loaded_resnext_model = self.appearance_branch.loaded_backbone.model_name
        self.loaded_tinyclip_model = self.semantic_branch.loaded_backbone.model_name

    def forward(self, x: torch.Tensor):
        f_app = self.appearance_branch(x)
        f_sem = self.semantic_branch(x)
        t_u = self.fusion_fc(torch.cat([f_app, f_sem], dim=1))
        t_s_prime = self.afem(t_u)
        t = t_u + t_s_prime
        t_bn = self.bnneck(t)
        t_bn_normalized = F.normalize(t_bn, p=2, dim=1)

        if self.training:
            logits = self.classifier(self.dropout(t_bn))
            return t_bn_normalized, logits

        return t_bn_normalized


def build_clip_senet(num_classes: int, **kwargs) -> CLIPSENet:
    """Build a CLIP-SENet model for M1 architecture validation."""

    return CLIPSENet(num_classes=num_classes, **kwargs)

In [ ]:
import json
import re
import time
import gc
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from torch.utils.data import DataLoader, Dataset

CAMERA_PATTERN = re.compile(r"^(?P<pid>-?\d+)_c(?P<camid>\d+)")
NUM_VERI_CAMERAS = 20

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

TRANSREID_IMG_SIZE = (224, 224)
CLIPSENET_IMG_SIZE = (320, 320)
TRANSREID_BATCH_SIZE = 64
CLIPSENET_BATCH_SIZE = 32
NUM_WORKERS = 4


def discover_veri_root() -> Path:
    for candidate in Path("/kaggle/input").rglob("*"):
        if candidate.is_dir() and (candidate / "image_query").is_dir() and (candidate / "image_test").is_dir():
            return candidate
    raise FileNotFoundError("VeRi-776 root not found under /kaggle/input")


def parse_veri_record(img_path: Path) -> dict:
    match = CAMERA_PATTERN.match(img_path.stem)
    if match is None:
        raise RuntimeError(f"Unexpected VeRi filename: {img_path.name}")
    parsed_camid = int(match.group("camid"))
    if not 1 <= parsed_camid <= NUM_VERI_CAMERAS:
        raise RuntimeError(f"Camera ID out of range for {img_path.name}: c{parsed_camid:03d}")
    return {
        "path": str(img_path),
        "pid": int(match.group("pid")),
        "camid": parsed_camid - 1,
        "sie_index": parsed_camid - 1,
    }


def parse_split(split_dir: Path):
    items = []
    pid_set = set()
    for img_path in sorted(split_dir.glob("*.jpg")):
        record = parse_veri_record(img_path)
        if record["pid"] == -1:
            continue
        pid_set.add(record["pid"])
        items.append(record)
    return items, len(pid_set)


class VeRiTensorDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        image = Image.open(item["path"]).convert("RGB")
        return self.transform(image), int(item["pid"]), int(item["camid"]), int(item["sie_index"]), item["path"]


def build_transform(size, mean, std):
    return T.Compose([
        T.Resize(size, interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


def build_tensor_loader(items, size, mean, std, batch_size):
    return DataLoader(
        VeRiTensorDataset(items, build_transform(size, mean, std)),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )


VERI_ROOT = discover_veri_root()
query_items, query_ids = parse_split(VERI_ROOT / "image_query")
gallery_items, gallery_ids = parse_split(VERI_ROOT / "image_test")

print("VERI_ROOT:", VERI_ROOT)
print(f"Query:   {len(query_items):,} images, {query_ids} IDs")
print(f"Gallery: {len(gallery_items):,} images, {gallery_ids} IDs")

In [ ]:
from src.stage2_features.transreid_model import build_transreid

SIE_NUM_CAMERAS = 20
CONCAT_PATCH_GEM_P = 3.0


def torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def find_file_under_input(filename: str, preferred_slug: str | None = None) -> Path:
    candidates = []
    for path in Path("/kaggle/input").rglob(filename):
        score = 100 if preferred_slug and preferred_slug in str(path) else 0
        candidates.append((score, path))
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    candidates.sort(key=lambda row: (-row[0], str(row[1])))
    for score, path in candidates[:10]:
        print(f"candidate {filename}: score={score:3d} {path}")
    return candidates[0][1]


def discover_clip_senet_checkpoint() -> Path:
    names = ("vehicle_clip_senet_veri776.pth", "best_mAP.pth", "best.pth")
    candidates = []
    for name in names:
        for path in Path("/kaggle/input").rglob(name):
            score = 0
            path_text = str(path)
            if "13-clip-senet-train" in path_text:
                score += 100
            if name == "vehicle_clip_senet_veri776.pth":
                score += 20
            if name == "best_mAP.pth":
                score += 10
            candidates.append((score, path))
    if not candidates:
        raise FileNotFoundError("Could not find CLIP-SENet v6 checkpoint from 13-clip-senet-train")
    candidates.sort(key=lambda row: (-row[0], str(row[1])))
    for score, path in candidates[:10]:
        print(f"candidate CLIP-SENet: score={score:3d} {path}")
    return candidates[0][1]


TRANSREID_WEIGHTS = find_file_under_input("vehicle_transreid_vit_base_veri776.pth", preferred_slug="mtmc-weights")
CLIPSENET_WEIGHTS = discover_clip_senet_checkpoint()


def load_transreid_model(weights_path: Path):
    model = build_transreid(
        num_classes=1,
        num_cameras=SIE_NUM_CAMERAS,
        embed_dim=768,
        vit_model="vit_base_patch16_clip_224.openai",
        pretrained=False,
        weights_path=str(weights_path),
        img_size=TRANSREID_IMG_SIZE,
    )
    model._concat_patch = False
    model._gem_p = CONCAT_PATCH_GEM_P
    return model.to(DEVICE).eval()


def load_clip_senet_model(weights_path: Path):
    payload = torch_load(weights_path)
    if isinstance(payload, dict) and "model_state" in payload:
        state_dict = payload["model_state"]
        checkpoint_kind = "payload:model_state"
    elif isinstance(payload, dict) and "model" in payload and isinstance(payload["model"], dict):
        state_dict = payload["model"]
        checkpoint_kind = "payload:model"
    elif isinstance(payload, dict) and payload and all(hasattr(value, "shape") for value in payload.values()):
        state_dict = payload
        checkpoint_kind = "state_dict"
    else:
        raise TypeError(f"Unsupported checkpoint format at {weights_path}: {type(payload).__name__}")
    classifier_weight = state_dict.get("classifier.weight")
    inferred_num_classes = int(classifier_weight.shape[0]) if classifier_weight is not None else 575
    model = build_clip_senet(num_classes=inferred_num_classes).to(DEVICE)
    missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
    if missing_keys or unexpected_keys:
        print("Missing keys:", missing_keys)
        print("Unexpected keys:", unexpected_keys)
        raise RuntimeError("CLIP-SENet checkpoint load was not strict")
    model.eval()
    return model, {"checkpoint_kind": checkpoint_kind, "num_classes": inferred_num_classes}


transreid_model = load_transreid_model(TRANSREID_WEIGHTS)
clipsenet_model, clipsenet_load_info = load_clip_senet_model(CLIPSENET_WEIGHTS)

MODEL_INFO = {
    "transreid_weights": str(TRANSREID_WEIGHTS),
    "clipsenet_weights": str(CLIPSENET_WEIGHTS),
    "clipsenet_load_info": clipsenet_load_info,
    "transreid_img_size": list(TRANSREID_IMG_SIZE),
    "clipsenet_img_size": list(CLIPSENET_IMG_SIZE),
}
print(json.dumps(MODEL_INFO, indent=2))

In [ ]:
@torch.no_grad()
def extract_transreid_features(model, items, mode: str):
    loader = build_tensor_loader(items, TRANSREID_IMG_SIZE, CLIP_MEAN, CLIP_STD, TRANSREID_BATCH_SIZE)
    features = []
    pids = []
    camids = []
    model.eval()
    model._concat_patch = mode == "concat_patch_flip"
    model._gem_p = CONCAT_PATCH_GEM_P
    started = time.time()
    for images, batch_pids, batch_camids, batch_sie, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        batch_sie = batch_sie.to(DEVICE, non_blocking=True).long()
        forward_views = []
        for view in (images, torch.flip(images, dims=[3])):
            output = model(view, cam_ids=batch_sie)
            if isinstance(output, (tuple, list)):
                output = output[-1]
            forward_views.append(F.normalize(output.float(), p=2, dim=1).cpu())
        batch_features = F.normalize(torch.stack(forward_views, dim=0).mean(dim=0), p=2, dim=1)
        features.append(batch_features.numpy().astype(np.float32))
        pids.append(batch_pids.numpy())
        camids.append(batch_camids.numpy())
    elapsed = time.time() - started
    merged = np.concatenate(features, axis=0)
    print(f"TransReID {mode}: {merged.shape} in {elapsed:.1f}s")
    model._concat_patch = False
    return merged, np.concatenate(pids), np.concatenate(camids)


@torch.no_grad()
def extract_clipsenet_features(model, items):
    loader = build_tensor_loader(items, CLIPSENET_IMG_SIZE, IMAGENET_MEAN, IMAGENET_STD, CLIPSENET_BATCH_SIZE)
    features = []
    pids = []
    camids = []
    started = time.time()
    model.eval()
    for images, batch_pids, batch_camids, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        output = model(images)
        if isinstance(output, (tuple, list)):
            output = output[-1]
        output = F.normalize(output.float(), p=2, dim=1)
        features.append(output.cpu().numpy().astype(np.float32))
        pids.append(batch_pids.numpy())
        camids.append(batch_camids.numpy())
    elapsed = time.time() - started
    merged = np.concatenate(features, axis=0)
    print(f"CLIP-SENet: {merged.shape} in {elapsed:.1f}s")
    return merged, np.concatenate(pids), np.concatenate(camids)


q_tr_768, q_pids, q_camids = extract_transreid_features(transreid_model, query_items, "single_flip")
g_tr_768, g_pids, g_camids = extract_transreid_features(transreid_model, gallery_items, "single_flip")
q_tr_1536, q_pids_1536, q_camids_1536 = extract_transreid_features(transreid_model, query_items, "concat_patch_flip")
g_tr_1536, g_pids_1536, g_camids_1536 = extract_transreid_features(transreid_model, gallery_items, "concat_patch_flip")
q_cs, q_pids_cs, q_camids_cs = extract_clipsenet_features(clipsenet_model, query_items)
g_cs, g_pids_cs, g_camids_cs = extract_clipsenet_features(clipsenet_model, gallery_items)

assert np.array_equal(q_pids, q_pids_1536) and np.array_equal(q_pids, q_pids_cs)
assert np.array_equal(g_pids, g_pids_1536) and np.array_equal(g_pids, g_pids_cs)
assert np.array_equal(q_camids, q_camids_1536) and np.array_equal(q_camids, q_camids_cs)
assert np.array_equal(g_camids, g_camids_1536) and np.array_equal(g_camids, g_camids_cs)

for name, array in {
    "query_transreid_768.npy": q_tr_768,
    "gallery_transreid_768.npy": g_tr_768,
    "query_transreid_1536.npy": q_tr_1536,
    "gallery_transreid_1536.npy": g_tr_1536,
    "query_clipsenet_2048.npy": q_cs,
    "gallery_clipsenet_2048.npy": g_cs,
    "query_pids.npy": q_pids,
    "query_camids.npy": q_camids,
    "gallery_pids.npy": g_pids,
    "gallery_camids.npy": g_camids,
} .items():
    np.save(FEATURE_DIR / name, array)

del transreid_model, clipsenet_model
gc.collect()
if DEVICE.startswith("cuda"):
    torch.cuda.empty_cache()

print("Saved feature arrays to", FEATURE_DIR)

In [ ]:
def l2_normalize(features):
    features = features.astype(np.float32, copy=False)
    return features / (np.linalg.norm(features, axis=1, keepdims=True) + 1e-12)


def compute_distance_from_similarity(similarity):
    return (1.0 - similarity).astype(np.float32, copy=False)


def eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids, max_rank=50):
    num_q, num_g = distmat.shape
    if num_g < max_rank:
        max_rank = num_g
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, np.newaxis]).astype(np.int32)
    all_cmc = []
    all_ap = []
    num_valid = 0
    for q_idx in range(num_q):
        q_pid = q_pids[q_idx]
        q_camid = q_camids[q_idx]
        order = indices[q_idx]
        remove = (g_pids[order] == q_pid) & (g_camids[order] == q_camid)
        keep = ~remove
        if not np.any(matches[q_idx][keep]):
            continue
        raw_cmc = matches[q_idx][keep]
        num_valid += 1
        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        num_rel = raw_cmc.sum()
        tmp_cmc = raw_cmc.cumsum()
        precision = tmp_cmc / (np.arange(len(tmp_cmc)) + 1.0)
        ap = (precision * raw_cmc.astype(bool)).sum() / num_rel if num_rel > 0 else 0.0
        all_ap.append(ap)
    if num_valid == 0:
        raise RuntimeError("No valid query found during VeRi evaluation")
    cmc = np.asarray(all_cmc, dtype=np.float32).mean(axis=0)
    return float(np.mean(all_ap)), cmc


def to_metric_dict(mAP, cmc):
    ranks = list(cmc)
    return {
        "mAP": float(mAP),
        "R1": float(ranks[min(0, len(ranks) - 1)]),
        "R5": float(ranks[min(4, len(ranks) - 1)]),
        "R10": float(ranks[min(9, len(ranks) - 1)]),
    }


def metric_sort_key(metrics):
    return (metrics["mAP"], metrics["R1"], metrics["R5"], metrics["R10"])


def print_metrics(label, metrics, duration_sec=None):
    suffix = "" if duration_sec is None else f" ({duration_sec:.1f}s)"
    print(
        f"{label}: mAP={metrics['mAP'] * 100:.4f}% "
        f"R1={metrics['R1'] * 100:.4f}% "
        f"R5={metrics['R5'] * 100:.2f}% "
        f"R10={metrics['R10'] * 100:.2f}%{suffix}"
    )


def average_query_expansion(features, k, iterations=1):
    current = l2_normalize(features.copy())
    if k <= 1:
        return current
    for _ in range(iterations):
        sim = current @ current.T
        topk = min(k, sim.shape[1])
        kth = max(topk - 1, 0)
        topk_idx = np.argpartition(-sim, kth=kth, axis=1)[:, :topk]
        expanded = np.zeros_like(current, dtype=np.float32)
        for index in range(current.shape[0]):
            expanded[index] = current[topk_idx[index]].mean(axis=0)
        current = l2_normalize(expanded)
    return current


@torch.no_grad()
def build_rerank_state_from_similarity(similarity, max_k1):
    sim_tensor = torch.as_tensor(similarity, dtype=torch.float32, device=DEVICE)
    original_dist = (2.0 - 2.0 * sim_tensor).clamp_min_(0).cpu().numpy().astype(np.float32)
    initial_rank = torch.topk(
        sim_tensor,
        k=min(max_k1 + 1, sim_tensor.shape[1]),
        dim=1,
        largest=True,
        sorted=True,
    ).indices.cpu().numpy().astype(np.int32)
    del sim_tensor
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
    return original_dist, initial_rank


def compute_reranking_torch(original_dist, initial_rank, query_num, k1=80, k2=15, lambda_value=0.2):
    all_num = original_dist.shape[0]
    V = np.zeros((all_num, all_num), dtype=np.float16)
    half_k1 = int(np.round(k1 / 2.0))
    for index in range(all_num):
        forward = initial_rank[index, :k1 + 1]
        backward = initial_rank[forward, :k1 + 1]
        reciprocal = forward[np.any(backward == index, axis=1)]
        reciprocal_expansion = reciprocal.copy()
        for candidate in reciprocal:
            candidate_forward = initial_rank[candidate, :half_k1 + 1]
            candidate_backward = initial_rank[candidate_forward, :half_k1 + 1]
            candidate_reciprocal = candidate_forward[np.any(candidate_backward == candidate, axis=1)]
            if candidate_reciprocal.size == 0:
                continue
            overlap = np.intersect1d(candidate_reciprocal, reciprocal)
            if overlap.size > (2.0 / 3.0) * candidate_reciprocal.size:
                reciprocal_expansion = np.concatenate((reciprocal_expansion, candidate_reciprocal))
        reciprocal_expansion = np.unique(reciprocal_expansion)
        weights = np.exp(-original_dist[index, reciprocal_expansion]).astype(np.float32)
        V[index, reciprocal_expansion] = (weights / (weights.sum() + 1e-12)).astype(np.float16)
    if k2 > 1:
        V_qe = np.zeros_like(V, dtype=np.float16)
        for index in range(all_num):
            V_qe[index] = V[initial_rank[index, :k2]].mean(axis=0)
        V = V_qe
    inv_index = [np.flatnonzero(V[:, column]) for column in range(all_num)]
    jaccard_dist = np.zeros((query_num, all_num), dtype=np.float32)
    for index in range(query_num):
        temp_min = np.zeros(all_num, dtype=np.float32)
        non_zero = np.flatnonzero(V[index])
        for nz in non_zero:
            related = inv_index[nz]
            temp_min[related] += np.minimum(np.float32(V[index, nz]), V[related, nz].astype(np.float32))
        jaccard_dist[index] = 1.0 - temp_min / (2.0 - temp_min)
    final_dist = jaccard_dist * (1.0 - lambda_value) + original_dist[:query_num] * lambda_value
    return final_dist[:, query_num:]


def evaluate_distmat(label, distmat, row):
    started = time.time()
    mAP, cmc = eval_market1501(distmat, q_pids, g_pids, q_camids, g_camids)
    metrics = to_metric_dict(mAP, cmc)
    metrics["duration_sec"] = float(time.time() - started)
    record = {**row, "label": label, "metrics": metrics}
    print_metrics(label, metrics, metrics["duration_sec"])
    return record

In [ ]:
RERANK_CANONICAL = {"k1": 80, "k2": 15, "lambda_value": 0.2}
AQE_K = 3
WEIGHTS = [round(i / 10, 1) for i in range(11)]
CONCAT_ALPHAS = [0.3, 0.5, 0.7]

q_tr_768 = l2_normalize(q_tr_768)
g_tr_768 = l2_normalize(g_tr_768)
q_tr_1536 = l2_normalize(q_tr_1536)
g_tr_1536 = l2_normalize(g_tr_1536)
q_cs = l2_normalize(q_cs)
g_cs = l2_normalize(g_cs)

feature_sets = {
    "transreid_768": (q_tr_768, g_tr_768),
    "transreid_1536": (q_tr_1536, g_tr_1536),
}


def score_similarity(q_tr, g_tr, q_cs, g_cs, w):
    return (w * (q_cs @ g_cs.T) + (1.0 - w) * (q_tr @ g_tr.T)).astype(np.float32)


def score_all_similarity(all_tr, all_cs, w):
    return (w * (all_cs @ all_cs.T) + (1.0 - w) * (all_tr @ all_tr.T)).astype(np.float32)


def concat_features(left, right, alpha):
    return l2_normalize(np.concatenate([alpha * left, (1.0 - alpha) * right], axis=1))


def evaluate_similarity_suite(strategy, params, q_features_or_sim, g_features=None, all_similarity=None):
    rows = []
    if g_features is None:
        qg_similarity = q_features_or_sim
    else:
        qg_similarity = q_features_or_sim @ g_features.T
    base_row = {"strategy": strategy, "params": params, "postprocess": "base", "aqe": None, "rerank": None}
    rows.append(evaluate_distmat(f"{strategy} {params} base", compute_distance_from_similarity(qg_similarity), base_row))

    if all_similarity is None:
        all_features = np.concatenate([q_features_or_sim, g_features], axis=0)
        all_similarity = (all_features @ all_features.T).astype(np.float32)

    started = time.time()
    state = build_rerank_state_from_similarity(all_similarity, max_k1=RERANK_CANONICAL["k1"])
    distmat = compute_reranking_torch(
        original_dist=state[0],
        initial_rank=state[1],
        query_num=len(q_pids),
        **RERANK_CANONICAL,
    )
    row = {"strategy": strategy, "params": params, "postprocess": "rerank", "aqe": None, "rerank": dict(RERANK_CANONICAL)}
    rows.append(evaluate_distmat(f"{strategy} {params} rerank", distmat, row))
    print(f"  rerank total wall={time.time() - started:.1f}s")
    return rows


def evaluate_feature_suite(strategy, params, q_features, g_features):
    rows = evaluate_similarity_suite(strategy, params, q_features, g_features=g_features)
    all_features = np.concatenate([q_features, g_features], axis=0)
    aqe_all = average_query_expansion(all_features, k=AQE_K, iterations=1)
    q_aqe = aqe_all[: len(q_features)]
    g_aqe = aqe_all[len(q_features) :]
    aqe_row = {"strategy": strategy, "params": params, "postprocess": "aqe", "aqe": {"k": AQE_K, "iterations": 1}, "rerank": None}
    rows.append(evaluate_distmat(f"{strategy} {params} AQE k={AQE_K}", compute_distance_from_similarity(q_aqe @ g_aqe.T), aqe_row))
    aqe_similarity = (aqe_all @ aqe_all.T).astype(np.float32)
    state = build_rerank_state_from_similarity(aqe_similarity, max_k1=RERANK_CANONICAL["k1"])
    distmat = compute_reranking_torch(
        original_dist=state[0],
        initial_rank=state[1],
        query_num=len(q_features),
        **RERANK_CANONICAL,
    )
    row = {"strategy": strategy, "params": params, "postprocess": "aqe_rerank", "aqe": {"k": AQE_K, "iterations": 1}, "rerank": dict(RERANK_CANONICAL)}
    rows.append(evaluate_distmat(f"{strategy} {params} AQE k={AQE_K}+rerank", distmat, row))
    return rows


def evaluate_score_suite(transreid_name, q_tr, g_tr, w):
    rows = []
    params = {"transreid": transreid_name, "w_clipsenet": float(w), "w_transreid": float(1.0 - w)}
    all_tr = np.concatenate([q_tr, g_tr], axis=0)
    all_cs = np.concatenate([q_cs, g_cs], axis=0)
    qg_similarity = score_similarity(q_tr, g_tr, q_cs, g_cs, w)
    all_similarity = score_all_similarity(all_tr, all_cs, w)
    rows.extend(evaluate_similarity_suite("score", params, qg_similarity, all_similarity=all_similarity))

    aqe_tr = average_query_expansion(all_tr, k=AQE_K, iterations=1)
    aqe_cs = average_query_expansion(all_cs, k=AQE_K, iterations=1)
    q_tr_aqe = aqe_tr[: len(q_tr)]
    g_tr_aqe = aqe_tr[len(q_tr) :]
    q_cs_aqe = aqe_cs[: len(q_cs)]
    g_cs_aqe = aqe_cs[len(q_cs) :]
    qg_aqe = score_similarity(q_tr_aqe, g_tr_aqe, q_cs_aqe, g_cs_aqe, w)
    aqe_row = {"strategy": "score", "params": params, "postprocess": "aqe", "aqe": {"k": AQE_K, "iterations": 1}, "rerank": None}
    rows.append(evaluate_distmat(f"score {params} AQE k={AQE_K}", compute_distance_from_similarity(qg_aqe), aqe_row))
    all_similarity_aqe = score_all_similarity(aqe_tr, aqe_cs, w)
    state = build_rerank_state_from_similarity(all_similarity_aqe, max_k1=RERANK_CANONICAL["k1"])
    distmat = compute_reranking_torch(
        original_dist=state[0],
        initial_rank=state[1],
        query_num=len(q_tr),
        **RERANK_CANONICAL,
    )
    row = {"strategy": "score", "params": params, "postprocess": "aqe_rerank", "aqe": {"k": AQE_K, "iterations": 1}, "rerank": dict(RERANK_CANONICAL)}
    rows.append(evaluate_distmat(f"score {params} AQE k={AQE_K}+rerank", distmat, row))
    return rows


results = []
experiment_start = time.time()

for transreid_name, (q_tr, g_tr) in feature_sets.items():
    for w in WEIGHTS:
        results.extend(evaluate_score_suite(transreid_name, q_tr, g_tr, w))
        gc.collect()
    for alpha in CONCAT_ALPHAS:
        q_concat = concat_features(q_tr, q_cs, alpha)
        g_concat = concat_features(g_tr, g_cs, alpha)
        params = {"transreid": transreid_name, "alpha_transreid": float(alpha), "alpha_clipsenet": float(1.0 - alpha)}
        results.extend(evaluate_feature_suite("concat", params, q_concat, g_concat))
        gc.collect()

# Extra parent drift checks requested by the locked spec. They do not affect the user-requested 11-weight canonical grid.
drift_checks = []
all_cs = np.concatenate([q_cs, g_cs], axis=0)
cs_aqe10 = average_query_expansion(all_cs, k=10, iterations=1)
state = build_rerank_state_from_similarity((cs_aqe10 @ cs_aqe10.T).astype(np.float32), max_k1=50)
distmat = compute_reranking_torch(state[0], state[1], len(q_pids), k1=50, k2=10, lambda_value=0.1)
drift_checks.append(evaluate_distmat(
    "drift clipsenet AQE k=10 rerank k1=50 k2=10 lambda=0.1",
    distmat,
    {"strategy": "drift", "params": {"parent": "clipsenet"}, "postprocess": "aqe10_rerank_clip_pref", "aqe": {"k": 10, "iterations": 1}, "rerank": {"k1": 50, "k2": 10, "lambda_value": 0.1}},
))
all_tr1536 = np.concatenate([q_tr_1536, g_tr_1536], axis=0)
tr_aqe3 = average_query_expansion(all_tr1536, k=3, iterations=1)
state = build_rerank_state_from_similarity((tr_aqe3 @ tr_aqe3.T).astype(np.float32), max_k1=80)
distmat = compute_reranking_torch(state[0], state[1], len(q_pids), k1=80, k2=15, lambda_value=0.2)
drift_checks.append(evaluate_distmat(
    "drift transreid_1536 AQE k=3 rerank k1=80 k2=15 lambda=0.2",
    distmat,
    {"strategy": "drift", "params": {"parent": "transreid_1536"}, "postprocess": "aqe3_rerank_09v", "aqe": {"k": 3, "iterations": 1}, "rerank": dict(RERANK_CANONICAL)},
))

all_rows = results + drift_checks
best_overall = max(results, key=lambda row: metric_sort_key(row["metrics"]))
concat_aqe_rows = [
    row for row in results
    if row["strategy"] == "concat" and row.get("aqe") and row["aqe"].get("k") == AQE_K
]
best_concat_aqe = max(concat_aqe_rows, key=lambda row: metric_sort_key(row["metrics"]))

target_metrics = best_concat_aqe["metrics"]
if target_metrics["mAP"] >= 0.9154 and target_metrics["R1"] >= 0.9833:
    verdict = "WIN"
elif target_metrics["mAP"] >= 0.905 or target_metrics["R1"] >= 0.980:
    verdict = "MARGINAL"
else:
    verdict = "FAIL"

summary = {
    "experiment": "14t_veri_fusion_clipsenet_transreid",
    "runtime_sec": float(time.time() - experiment_start),
    "verdict": verdict,
    "verdict_basis": "best concat row with AQE k=3, including AQE-only and AQE+rerank rows",
    "best_concat_aqe": best_concat_aqe,
    "best_overall": best_overall,
    "top10": sorted(results, key=lambda row: metric_sort_key(row["metrics"]), reverse=True)[:10],
    "drift_checks": drift_checks,
    "thresholds": {"WIN": {"mAP": 0.9154, "R1": 0.9833}, "MARGINAL": {"mAP": 0.905, "R1": 0.980}},
}

recipe = {
    "models": MODEL_INFO,
    "dataset": "abhyudaya12/veri-vehicle-re-identification-dataset",
    "score_fusion": {"w_clipsenet_values": WEIGHTS, "formula": "w*S_clipsenet + (1-w)*S_transreid"},
    "concat_fusion": {"alpha_transreid_values": CONCAT_ALPHAS, "formula": "L2([alpha*transreid, (1-alpha)*clipsenet])"},
    "postprocess_suite": ["base", "rerank", "aqe_k3", "aqe_k3_rerank"],
    "rerank_canonical": RERANK_CANONICAL,
    "aqe_k": AQE_K,
    "query_count": int(len(q_pids)),
    "gallery_count": int(len(g_pids)),
}

for path, payload in {
    WORKING / "eval_results.json": {"rows": all_rows},
    WORKING / "14t_fusion_results.json": {"rows": all_rows},
    WORKING / "14t_summary.json": summary,
    WORKING / "recipe.json": recipe,
}.items():
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("Wrote", path)

print(json.dumps(summary, indent=2))